In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("EcommerceDataPipeline").getOrCreate()

Read the Data From Bronze Layer


In [0]:
df = spark.table("workspace.bronze_ecom.users")
df.display()

Normalize Country code to uppercase

In [0]:
df = df.withColumn('countryCode',upper(col('countryCode')))
df.select('countryCode').display()

Adding language Abbreviations


In [0]:
# df = df.withColumn('language_full',
#                    expr("CASE WHEN language = 'EN' THEN 'English' " +
#                         "WHEN language ='FR' THEN 'French' " +
#                         "ELSE 'OTHER' END"
#                         ))


df = df.withColumn('language_full',
                   expr("""
                        CASE WHEN UPPER(language) = 'EN' THEN 'English'
                             WHEN UPPER(language) ='FR' THEN 'French'
                             ELSE 'Other'
                         END
                        """))

In [0]:
df.display()

Converting "M -> Male and F -> Female"


In [0]:
# df = df.withColumn("gender",
#               when(col('gender').startswith("M"),"Male")
#               .when(col('gender').startswith("F"),"Female")
#               .otherwise("other"))
# df.display()

df = df.withColumn('gender',
                   when(upper(col('gender')).startswith('M'),"Male")\
                       .when(upper(col('gender')).startswith('F'),'Female')\
                           .otherwise("other"))

Using regexp_replace to clean civilitytitle values

In [0]:
df = df.withColumn('civilitytitle_clean',
                   regexp_replace(col('civilitytitle'),'(?i)(Mme|Ms|Mrs)',"Ms")
                   )


Derive the New column ' years_since_last_login' from 'dayssincelastlogin'

In [0]:

df = df.withColumn('years_since_last_login',
                   round(col('daysSinceLastLogin')/365,2))


Calculate the age of account in years and categorize into 'account-age-group'

In [0]:
df = df.withColumn('account_age_years',round(coalesce(col('seniority'),lit(0)) /365,2))



In [0]:

df =df.withColumn("account_age_group",
                             when(col("account_age_years") < 1, "New")
                             .when((col("account_age_years") >= 1) & (col("account_age_years") < 3), "Intermediate")
                             .otherwise("Experienced"))


In [0]:
df.display()

Add  current_year column

In [0]:
df = df.withColumn('current_year',year(current_date()))

In [0]:
from pyspark.sql.functions import col, lit, concat, substring

# df = df.withColumn(
#     "user_descriptor",
#     concat(
#         col("gender"), lit("_"),
#         col("countryCode"), lit("_"),
#         substring(col("civilitytitle_clean"), 1, 3), lit("_"),
#         col("language_full")
#     )
# )

df = df.withColumn(
    "user_descriptor",
    concat(
        coalesce(col("gender"), lit("Unknown")), lit("_"),
        coalesce(col("countrycode"), lit("XX")), lit("_"),
        coalesce(substring(col("civilitytitle_clean"), 1, 3), lit("NA")), lit("_"),
        coalesce(col("language_full"), lit("Other"))
    )
)
                   
                   
                   

df.display()

In [0]:
df = df.withColumn("flag_long_title",length(col('civilitytitle')) > 10)
df.display()

In [0]:
df = df.withColumn("hasanyapp",col('hasanyapp').cast('boolean'))
df = df.withColumn("hasandroidapp",col("hasandroidapp").cast('boolean'))
df = df.withColumn("hasiosapp",col('hasiosapp').cast('boolean'))
df = df.withColumn('hasprofilepicture',col('hasprofilepicture').cast('boolean'))

df = df.withColumn('socialnbfollowers',col('socialnbfollowers').cast(IntegerType()))
df = df.withColumn('socialnbfollows',col('socialnbfollows').cast(IntegerType()))

df = df.withColumn('productspassrate',col('productspassrate').cast(DecimalType(10,2)))

df = df.withColumn('seniorityasmonths',col('seniorityasmonths').cast(DecimalType(10,2)))

df = df.withColumn('seniorityasyears',col('seniorityasyears').cast(DecimalType(10,2)))

df.display()

In [0]:
df = df.withColumn("dayssincelastlogin",
                             when(col("dayssincelastlogin").isNotNull(),
                                  col("dayssincelastlogin").cast(IntegerType()))
                             .otherwise(0))

# **_Write the users data into silver Table
# _**

In [0]:
df = (
    df.write.mode('overwrite').format('delta').saveAsTable('silver_ecom.users')
)

# **_Read the Buyers data_**


In [0]:
buyers_df = spark.table("workspace.bronze_ecom.buyers")
buyers_df.display()

In [0]:
integer_columns = [
    'buyers', 'topbuyers', 'femalebuyers', 'malebuyers',
    'topfemalebuyers', 'topmalebuyers', 'totalproductsbought',
    'totalproductswished', 'totalproductsliked', 'toptotalproductsbought',
    'toptotalproductswished', 'toptotalproductsliked'
]

for column_name in integer_columns:
    buyers_df = buyers_df.withColumn(column_name, col(column_name).cast(IntegerType()))

CASTING DECIMAL VALUES


In [0]:
# Casting Decimal columns
decimal_columns = [
    'topbuyerratio', 'femalebuyersratio', 'topfemalebuyersratio',
    'boughtperwishlistratio', 'boughtperlikeratio', 'topboughtperwishlistratio',
    'topboughtperlikeratio', 'meanproductsbought', 'meanproductswished',
    'meanproductsliked', 'topmeanproductsbought', 'topmeanproductswished',
    'topmeanproductsliked', 'meanofflinedays', 'topmeanofflinedays',
    'meanfollowers', 'meanfollowing', 'topmeanfollowers', 'topmeanfollowing'
]

for column_name in decimal_columns:
    buyers_df = buyers_df.withColumn(column_name, col(column_name).cast(DecimalType(10, 2)))

In [0]:

buyers_df.printSchema()

Normalize Country Names

In [0]:
#Normalize country names
buyers_df = buyers_df.withColumn("country", initcap(col("country")))

for col_name in integer_columns:
   buyers_df = buyers_df.fillna({col_name: 0})

# Calculate the ratio of female to male buyers
buyers_df = buyers_df.withColumn("female_to_male_ratio", 
                               round(col("femalebuyers") / (col("malebuyers") + 1), 2))

# Determine the market potential by comparing wishlist and purchases
buyers_df = buyers_df.withColumn("wishlist_to_purchase_ratio", 
                               round(col("totalproductswished") / (col("totalproductsbought") + 1), 2))

# Tag countries with a high engagement ratio
high_engagement_threshold = 0.5
buyers_df = buyers_df.withColumn("high_engagement",
                               when(col("boughtperwishlistratio") > high_engagement_threshold, True)
                               .otherwise(False))
                               
    # Flag markets with increasing female buyer participation
buyers_df = buyers_df.withColumn("growing_female_market",
                               when(col("femalebuyersratio") > col("topfemalebuyersratio"), True)
                               .otherwise(False))




In [0]:
buyers_df.printSchema()


In [0]:

buyers_df.write.mode('overwrite').format('delta').saveAsTable('silver_ecom.buyers')


In [0]:
sellers_df = spark.table('workspace.bronze_ecom.sellers')
sellers_df.display()

In [0]:
sellers_df = sellers_df \
    .withColumn("nbsellers", col("nbsellers").cast(IntegerType())) \
    .withColumn("meanproductssold", col("meanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("meanproductslisted", col("meanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meansellerpassrate", col("meansellerpassrate").cast(DecimalType(10, 2))) \
    .withColumn("totalproductssold", col("totalproductssold").cast(IntegerType())) \
    .withColumn("totalproductslisted", col("totalproductslisted").cast(IntegerType())) \
    .withColumn("meanproductsbought", col("meanproductsbought").cast(DecimalType(10, 2))) \
    .withColumn("meanproductswished", col("meanproductswished").cast(DecimalType(10, 2))) \
    .withColumn("meanproductsliked", col("meanproductsliked").cast(DecimalType(10, 2))) \
    .withColumn("totalbought", col("totalbought").cast(IntegerType())) \
    .withColumn("totalwished", col("totalwished").cast(IntegerType())) \
    .withColumn("totalproductsliked", col("totalproductsliked").cast(IntegerType())) \
    .withColumn("meanfollowers", col("meanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("meanfollows", col("meanfollows").cast(DecimalType(10, 2))) \
    .withColumn("percentofappusers", col("percentofappusers").cast(DecimalType(10, 2))) \
    .withColumn("percentofiosusers", col("percentofiosusers").cast(DecimalType(10, 2))) \
    .withColumn("meanseniority", col("meanseniority").cast(DecimalType(10, 2)))
     

Normalize country names and gender values

In [0]:
sellers_df = sellers_df.withColumn('country',initcap(col('country'))).withColumn('sex',upper(col('sex')))

**_Add a column to categorize the nimber of sellers_**

In [0]:
sellers_df = sellers_df.withColumn('seller_size_category',
                                   when(col('nbsellers') < 500,'small')\
                                       .when((col('nbsellers') >= 500) & (col('nbsellers') < 2000),'Medium')\
                                           .otherwise('Large'))

Calculate the mean products listed per se;;er as am indicator of seller activity


In [0]:
sellers_df = sellers_df.withColumn('mean_products_listed_per_seller',
                                   round(col('totalproductslisted')/col('nbsellers'),2))

Indentify markets with high sellers pass rate

In [0]:
sellers_df = sellers_df.withColumn('high_seller_pass_rate',
                                   when(col('meansellerpassrate') > 0.75,'High')\
                                       .otherwise("Normal"))
                                       

In [0]:
mean_pass_rate = sellers_df.select(round(avg("meansellerpassrate"), 2).alias("avg_pass_rate")).collect()[0]["avg_pass_rate"]

sellersDF = sellers_df.withColumn("meansellerpassrate",
                                 when(col("meansellerpassrate").isNull(), mean_pass_rate)
                                 .otherwise(col("meansellerpassrate")))

In [0]:
sellers_df.write.mode('overwrite').format('delta').saveAsTable('silver_ecom.sellers')

In [0]:

countries_df = spark.table('workspace.bronze_ecom.countries')

countries_df = countries_df \
    .withColumn("sellers", col("sellers").cast(IntegerType())) \
    .withColumn("topsellers", col("topsellers").cast(IntegerType())) \
    .withColumn("topsellerratio", col("topsellerratio").cast(DecimalType(10, 2))) \
    .withColumn("femalesellersratio", col("femalesellersratio").cast(DecimalType(10, 2))) \
    .withColumn("topfemalesellersratio", col("topfemalesellersratio").cast(DecimalType(10, 2))) \
    .withColumn("femalesellers", col("femalesellers").cast(IntegerType())) \
    .withColumn("malesellers", col("malesellers").cast(IntegerType())) \
    .withColumn("topfemalesellers", col("topfemalesellers").cast(IntegerType())) \
    .withColumn("topmalesellers", col("topmalesellers").cast(IntegerType())) \
    .withColumn("countrysoldratio", col("countrysoldratio").cast(DecimalType(10, 2))) \
    .withColumn("bestsoldratio", col("bestsoldratio").cast(DecimalType(10, 2))) \
    .withColumn("toptotalproductssold", col("toptotalproductssold").cast(IntegerType())) \
    .withColumn("totalproductssold", col("totalproductssold").cast(IntegerType())) \
    .withColumn("toptotalproductslisted", col("toptotalproductslisted").cast(IntegerType())) \
    .withColumn("totalproductslisted", col("totalproductslisted").cast(IntegerType())) \
    .withColumn("topmeanproductssold", col("topmeanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("topmeanproductslisted", col("topmeanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meanproductssold", col("meanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("meanproductslisted", col("meanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meanofflinedays", col("meanofflinedays").cast(DecimalType(10, 2))) \
    .withColumn("topmeanofflinedays", col("topmeanofflinedays").cast(DecimalType(10, 2))) \
    .withColumn("meanfollowers", col("meanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("meanfollowing", col("meanfollowing").cast(DecimalType(10, 2))) \
    .withColumn("topmeanfollowers", col("topmeanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("topmeanfollowing", col("topmeanfollowing").cast(DecimalType(10, 2)))

countries_df = countries_df.withColumn("country", initcap(col("country")))


# Calculating the ratio of top sellers to total sellers
countries_df = countries_df.withColumn("top_seller_ratio", 
                                        round(col("topsellers") / col("sellers"), 2))

# countriesDF countries with a high ratio of female sellers
countries_df = countries_df.withColumn("high_female_seller_ratio", 
                                        when(col("femalesellersratio") > 0.5, True).otherwise(False))

# Adding a performance indicator based on the sold/listed ratio
countries_df = countries_df.withColumn("performance_indicator", 
                                        round(col("toptotalproductssold") / (col("toptotalproductslisted") + 1), 2))

# Flag countries with exceptionally high performance
performance_threshold = 0.8
countries_df = countries_df.withColumn("high_performance", 
                                        when(col("performance_indicator") > performance_threshold, True).otherwise(False))

countries_df = countries_df.withColumn("activity_level",
                                       when(col("meanofflinedays") < 30, "Highly Active")
                                       .when((col("meanofflinedays") >= 30) & (col("meanofflinedays") < 60), "Moderately Active")
                                       .otherwise("Low Activity"))


countries_df.write.format("delta").mode("overwrite").saveAsTable('silver_ecom.countries')

identify